#  Micok-Homes Analysis
### Full Property Data Analysis — Dashboard, Charts & Excel Export


## Step 1 — Install Required Libraries
*Only needed the first time. Safe to skip if already installed.*

In [ ]:
# Run this once to install all required libraries
# If already installed, nothing will change
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "pandas", "openpyxl", "matplotlib", "seaborn", "-q"])
print(" All libraries ready!")

## Step 2 — Import Libraries & Configuration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage
import warnings, os
warnings.filterwarnings('ignore')

# ── FILE PATHS ── Change these if your files are in a different folder
FILE_PATH = 'Micok-Homes_Dataset.xlsx'   # Your input dataset
OUT_XLSX  = 'Micok-Homes_Analysis.xlsx'  # Output Excel file name
CHART_DIR = 'charts'                      # Folder where charts are saved
os.makedirs(CHART_DIR, exist_ok=True)

# ── COLOUR PALETTE ──
BLUE   = "1F4E79"
TEAL   = "00B0F0"
ORANGE = "ED7D31"
GREEN  = "70AD47"
PURPLE = "7030A0"
RED    = "C00000"
GREY   = "F2F2F2"
WHITE  = "FFFFFF"
GOLD   = "FFD700"

print(" Libraries imported and config set!")

## Step 3 — Load & Preview the Dataset

In [ ]:
df = pd.read_excel(FILE_PATH)
df.columns = df.columns.str.strip()

print(f" Loaded {len(df):,} rows and {len(df.columns)} columns")
print(f"   Columns: {list(df.columns)}")
df.head()

## Step 4 — Date Feature Engineering

In [ ]:
df['Listing Date']   = pd.to_datetime(df['Listing Date'], dayfirst=True)
df['Day']            = df['Listing Date'].dt.day
df['Day of Week']    = df['Listing Date'].dt.day_name()
df['Week']           = df['Listing Date'].dt.isocalendar().week.astype(int)
df['Month']          = df['Listing Date'].dt.month
df['Month Name']     = df['Listing Date'].dt.month_name()
df['Quarter']        = df['Listing Date'].dt.quarter
df['Quarter Label']  = 'Q' + df['Quarter'].astype(str) + ' ' + df['Listing Date'].dt.year.astype(str)
df['Year']           = df['Listing Date'].dt.year
df['Month Year']     = df['Listing Date'].dt.to_period('M').astype(str)
df['Days on Market'] = (pd.Timestamp.today() - df['Listing Date']).dt.days
df['Property Age']   = df['Year'] - df['Year Built']

print(" Date features added!")
df[['Listing Date','Day of Week','Quarter Label','Days on Market','Property Age']].head()

## Step 5 — Create Data Filters

In [ ]:
filters = {
    'Lagos':             df[df['Location'] == 'Lagos'],
    'Abuja':             df[df['Location'] == 'Abuja'],
    'Port Harcourt':     df[df['Location'] == 'Port Harcourt'],
    'Kano':              df[df['Location'] == 'Kano'],
    'Ibadan':            df[df['Location'] == 'Ibadan'],
    'Duplex':            df[df['Property Type'] == 'Duplex'],
    'Flat':              df[df['Property Type'] == 'Flat'],
    'Mansion':           df[df['Property Type'] == 'Mansion'],
    'Bungalow':          df[df['Property Type'] == 'Bungalow'],
    'Studio Apartment':  df[df['Property Type'] == 'Studio Apartment'],
    'Affordable <10M':   df[df['Price (NGN)'] < 10_000_000],
    'Mid 10M-50M':       df[(df['Price (NGN)'] >= 10_000_000) & (df['Price (NGN)'] < 50_000_000)],
    'Upper 50M-200M':    df[(df['Price (NGN)'] >= 50_000_000) & (df['Price (NGN)'] < 200_000_000)],
    'Luxury >200M':      df[df['Price (NGN)'] >= 200_000_000],
    'Available':         df[df['Status'] == 'Available'],
    'Sold':              df[df['Status'] == 'Sold'],
    'Rented':            df[df['Status'] == 'Rented'],
    'Under Offer':       df[df['Status'] == 'Under Offer'],
    '1 Bed':             df[df['Bedrooms'] == 1],
    '3+ Bedrooms':       df[df['Bedrooms'] >= 3],
    '5+ Bedrooms':       df[df['Bedrooms'] >= 5],
    '2022 Listings':     df[df['Year'] == 2022],
    '2023 Listings':     df[df['Year'] == 2023],
    '2024 Listings':     df[df['Year'] == 2024],
    'Q1 Listings':       df[df['Quarter'] == 1],
    'Q2 Listings':       df[df['Quarter'] == 2],
    'Q3 Listings':       df[df['Quarter'] == 3],
    'Q4 Listings':       df[df['Quarter'] == 4],
    'Weekend Listings':  df[df['Day of Week'].isin(['Saturday', 'Sunday'])],
    'Weekday Listings':  df[~df['Day of Week'].isin(['Saturday', 'Sunday'])],
    'Has Parking':       df[df['Parking Spaces'] >= 1],
    'No Parking':        df[df['Parking Spaces'] == 0],
    '2+ Parking':        df[df['Parking Spaces'] >= 2],
    'New Build <5yrs':   df[df['Property Age'] < 5],
    'Old Build >20yrs':  df[df['Property Age'] > 20],
    'Lagos Premium':     df[(df['Location']=='Lagos') & (df['Bedrooms']>=3) &
                            (df['Status']=='Available') & (df['Price (NGN)']<=100_000_000)],
    'Abuja Luxury':      df[(df['Location']=='Abuja') & (df['Price (NGN)']>=100_000_000)],
    'Large Mansions':    df[(df['Property Type']=='Mansion') & (df['Size (sqm)']>=300)],
}

print(f" {len(filters)} filters created!")
for name, fdf in filters.items():
    print(f"   {name}: {len(fdf):,} records")

## Step 6 — Analysis Tables

In [ ]:
avg_loc = (
    df.groupby('Location')['Price (NGN)']
    .agg(Count='count', Mean='mean', Median='median', Min='min', Max='max')
    .round(0).sort_values('Mean', ascending=False).reset_index()
)
avg_type = (
    df.groupby('Property Type')['Price (NGN)']
    .agg(Count='count', Mean='mean', Median='median')
    .round(0).sort_values('Mean', ascending=False).reset_index()
)
listings_year  = df.groupby('Year').size().reset_index(name='Count')
listings_month = df.groupby(['Year','Month','Month Name']).size().reset_index(name='Count').sort_values(['Year','Month'])
listings_qtr   = df.groupby('Quarter Label').size().reset_index(name='Count')
status_df      = df['Status'].value_counts().reset_index()
status_df.columns = ['Status', 'Count']
agent_df = (
    df.groupby('Agent').agg(
        Listings=('Property ID','count'),
        Avg_Price=('Price (NGN)','mean'),
        Total_Value=('Price (NGN)','sum')
    ).round(0).sort_values('Total_Value', ascending=False).reset_index()
)
top10 = df.nlargest(10,'Price (NGN)')[
    ['Property ID','Location','Property Type','Price (NGN)','Bedrooms','Status','Agent']
].reset_index(drop=True)
df['Price Segment'] = pd.cut(
    df['Price (NGN)'],
    bins=[0, 10_000_000, 50_000_000, 100_000_000, 200_000_000, float('inf')],
    labels=['<10M','10M-50M','50M-100M','100M-200M','>200M']
)
segment_df = df['Price Segment'].value_counts().sort_index().reset_index()
segment_df.columns = ['Segment','Count']
df['Price/sqm'] = (df['Price (NGN)'] / df['Size (sqm)']).round(0)
price_sqm = df.groupby('Location')['Price/sqm'].mean().round(0).sort_values(ascending=False).reset_index()

print(" Analysis tables ready!")
print("\n Average Price by Location:")
display(avg_loc)

## Step 7 — Pivot Tables

In [ ]:
pivot1 = pd.pivot_table(df, values='Price (NGN)', index='Location',
                        columns='Property Type', aggfunc='count', fill_value=0).reset_index()
pivot2 = pd.pivot_table(df, values='Price (NGN)', index='Location',
                        columns='Property Type', aggfunc='mean', fill_value=0).round(0).reset_index()
pivot3 = pd.pivot_table(df, values='Price (NGN)', index='Year',
                        columns='Quarter', aggfunc='count', fill_value=0).reset_index()
pivot3.columns = ['Year'] + [f'Q{c}' for c in pivot3.columns[1:]]
pivot4 = pd.pivot_table(df, values='Price (NGN)', index='Year',
                        columns='Status', aggfunc='mean', fill_value=0).round(0).reset_index()
pivot5 = pd.pivot_table(df, values='Price (NGN)', index='Location',
                        columns='Bedrooms', aggfunc='count', fill_value=0).reset_index()
pivot6 = pd.pivot_table(df, values='Price (NGN)', index='Property Type',
                        columns='Status', aggfunc='count', fill_value=0).reset_index()
pivot7 = pd.pivot_table(df, values='Price/sqm', index='Location',
                        columns='Property Type', aggfunc='mean', fill_value=0).round(0).reset_index()

print(" 7 pivot tables created!")
print("\n Pivot 1 — Listing Count: Location x Property Type:")
display(pivot1)

## Step 8 — Generate All Charts
*Charts are saved to the `charts/` folder and will also display here in the notebook.*

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')

def save_chart(name):
    path = f"{CHART_DIR}/{name}"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    return path

charts = {}

# 1. Price Distribution
fig, ax = plt.subplots(figsize=(11,5))
sns.histplot(df['Price (NGN)']/1e6, bins=35, color='#1F4E79', kde=True, ax=ax)
ax.set_title('Property Price Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Price (NGN Millions)'); ax.set_ylabel('Count')
charts['price_dist'] = save_chart('01_price_distribution.png')

# 2. Count by Location
fig, ax = plt.subplots(figsize=(10,5))
lc = df['Location'].value_counts()
bars = ax.barh(lc.index, lc.values, color='#00B0F0')
ax.bar_label(bars, padding=3)
ax.set_title('Property Count by Location', fontsize=14, fontweight='bold')
ax.set_xlabel('Count')
charts['by_loc'] = save_chart('02_count_by_location.png')

# 3. Avg Price by Location
fig, ax = plt.subplots(figsize=(10,5))
al = df.groupby('Location')['Price (NGN)'].mean().sort_values()/1e6
bars = ax.barh(al.index, al.values, color='#ED7D31')
ax.bar_label(bars, fmt='%.1fM', padding=3)
ax.set_title('Average Price by Location (NGN Millions)', fontsize=14, fontweight='bold')
charts['avg_loc'] = save_chart('03_avg_price_location.png')

# 4. Monthly Trend
fig, ax = plt.subplots(figsize=(13,5))
monthly = df.groupby('Month Year').size().reset_index(name='Count').sort_values('Month Year')
ax.plot(monthly['Month Year'], monthly['Count'], marker='o', color='#1F4E79', linewidth=2)
ax.fill_between(range(len(monthly)), monthly['Count'], alpha=0.15, color='#1F4E79')
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['Month Year'], rotation=45, ha='right', fontsize=8)
ax.set_title('Monthly Listings Trend', fontsize=14, fontweight='bold')
ax.set_ylabel('Listings')
charts['monthly'] = save_chart('04_monthly_trend.png')

# 5. Property Type Pie
fig, ax = plt.subplots(figsize=(8,7))
tc = df['Property Type'].value_counts()
colors_list = ['#1F4E79','#00B0F0','#ED7D31','#70AD47','#7030A0','#C00000','#FFD700']
ax.pie(tc, labels=tc.index, autopct='%1.1f%%', startangle=140, colors=colors_list)
ax.set_title('Property Type Distribution', fontsize=14, fontweight='bold')
charts['type_pie'] = save_chart('05_property_type_pie.png')

# 6. Status Bar
fig, ax = plt.subplots(figsize=(8,4))
sc = df['Status'].value_counts()
bars = ax.bar(sc.index, sc.values, color=['#70AD47','#C00000','#00B0F0','#ED7D31'])
ax.bar_label(bars, padding=3)
ax.set_title('Listings by Status', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
charts['status'] = save_chart('06_status.png')

# 7. Avg Price by Property Type
fig, ax = plt.subplots(figsize=(11,5))
at = df.groupby('Property Type')['Price (NGN)'].mean().sort_values(ascending=False)/1e6
bars = ax.bar(at.index, at.values, color='#7030A0')
ax.bar_label(bars, fmt='%.1fM', padding=3, fontsize=8)
ax.set_title('Average Price by Property Type (NGN Millions)', fontsize=14, fontweight='bold')
ax.set_ylabel('Avg Price (NGN Millions)')
ax.set_xticklabels(at.index, rotation=20, ha='right')
charts['avg_type'] = save_chart('07_avg_price_type.png')

# 8. Heatmap
fig, ax = plt.subplots(figsize=(9,5))
hd = df.pivot_table(values='Property ID', index='Year', columns='Quarter', aggfunc='count', fill_value=0)
sns.heatmap(hd, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Listings Heatmap: Year x Quarter', fontsize=14, fontweight='bold')
charts['heatmap'] = save_chart('08_heatmap_quarterly.png')

# 9. Boxplot
fig, ax = plt.subplots(figsize=(12,5))
order = df.groupby('Location')['Price (NGN)'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Location', y='Price (NGN)', order=order, palette='Set2', ax=ax)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.0f}M'))
ax.set_title('Price Distribution by Location', fontsize=14, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
charts['boxplot'] = save_chart('09_boxplot_location.png')

# 10. Price Segments
fig, ax = plt.subplots(figsize=(9,4))
bars = ax.bar(segment_df['Segment'], segment_df['Count'], color='#00B0F0')
ax.bar_label(bars, padding=3)
ax.set_title('Listings by Price Segment', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
charts['segment'] = save_chart('10_price_segments.png')

# 11. Agent Performance
fig, ax = plt.subplots(figsize=(11,5))
top_agents = agent_df.head(10)
bars = ax.barh(top_agents['Agent'], top_agents['Total_Value']/1e6, color='#ED7D31')
ax.bar_label(bars, fmt='%.0fM', padding=3, fontsize=8)
ax.set_title('Top 10 Agents by Total Portfolio Value (NGN Millions)', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Value (NGN Millions)')
charts['agents'] = save_chart('11_agent_performance.png')

# 12. Bedrooms vs Price
fig, ax = plt.subplots(figsize=(8,5))
bed_price = df.groupby('Bedrooms')['Price (NGN)'].mean()/1e6
ax.plot(bed_price.index, bed_price.values, marker='o', color='#1F4E79', linewidth=2.5, markersize=8)
ax.fill_between(bed_price.index, bed_price.values, alpha=0.15, color='#1F4E79')
for x, y in zip(bed_price.index, bed_price.values):
    ax.annotate(f'{y:.1f}M', (x,y), textcoords='offset points', xytext=(0,10), ha='center', fontsize=9)
ax.set_title('Average Price by Number of Bedrooms', fontsize=14, fontweight='bold')
ax.set_xlabel('Bedrooms'); ax.set_ylabel('Avg Price (NGN Millions)')
ax.set_xticks(bed_price.index)
charts['bedrooms'] = save_chart('12_bedrooms_vs_price.png')

# 13. Price per sqm
fig, ax = plt.subplots(figsize=(10,5))
bars = ax.barh(price_sqm['Location'], price_sqm['Price/sqm']/1000, color='#70AD47')
ax.bar_label(bars, fmt='%.0fK/sqm', padding=3, fontsize=9)
ax.set_title('Average Price per sqm by Location (NGN Thousands)', fontsize=14, fontweight='bold')
ax.set_xlabel('NGN per sqm (Thousands)')
charts['price_sqm'] = save_chart('13_price_per_sqm.png')

# 14. Yearly Listings
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(listings_year['Year'].astype(str), listings_year['Count'], color='#1F4E79', width=0.5)
ax.bar_label(bars, padding=3)
ax.set_title('Total Listings per Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
charts['yearly'] = save_chart('14_yearly_listings.png')

print(f"\n✅ {len(charts)} charts generated and saved to '{CHART_DIR}/' folder!")

## Step 9 — Build & Export Excel Workbook
*This creates the final `Micok-Homes_Analysis.xlsx` file with all 8 sheets.*

In [ ]:
wb = Workbook()

def write_df(ws, dataframe, start_row=1, header_bg=BLUE):
    for ci, col_name in enumerate(dataframe.columns, 1):
        c = ws.cell(row=start_row, column=ci, value=str(col_name))
        c.font = Font(bold=True, color=WHITE, size=10)
        c.fill = PatternFill('solid', start_color=header_bg)
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    for ri, row_data in enumerate(dataframe.itertuples(index=False), start_row+1):
        for ci, val in enumerate(row_data, 1):
            val = int(val) if hasattr(val, 'item') else val
            c = ws.cell(row=ri, column=ci, value=val)
            c.alignment = Alignment(horizontal='center')
            if ri % 2 == 0:
                c.fill = PatternFill('solid', start_color=GREY)
    for col in ws.columns:
        max_len = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 42)

def section_title(ws, row, col, text, color=BLUE):
    c = ws.cell(row=row, column=col, value=text)
    c.font = Font(bold=True, size=12, color=color)

# ── Sheet 1: Dashboard ──
ws1 = wb.active
ws1.title = "Dashboard"
ws1.sheet_view.showGridLines = False
ws1.row_dimensions[1].height = 40
ws1['A1'] = "MICOK-HOMES ANALYSIS DASHBOARD"
ws1['A1'].font = Font(bold=True, size=18, color=BLUE)
ws1.merge_cells('A1:I1')
ws1['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws1['A1'].fill = PatternFill('solid', start_color='DAEEF3')

kpis = [
    ("Total Listings",  f"{len(df):,}",                                BLUE),
    ("Locations",       df['Location'].nunique(),                       TEAL),
    ("Property Types",  df['Property Type'].nunique(),                  ORANGE),
    ("Available",       f"{len(df[df['Status']=='Available']):,}",      GREEN),
    ("Sold",            f"{len(df[df['Status']=='Sold']):,}",           RED),
    ("Rented",          f"{len(df[df['Status']=='Rented']):,}",         PURPLE),
    ("Avg Price",       f"NGN {df['Price (NGN)'].mean()/1e6:,.1f}M",   BLUE),
    ("Median Price",    f"NGN {df['Price (NGN)'].median()/1e6:,.1f}M", TEAL),
    ("Highest Price",   f"NGN {df['Price (NGN)'].max()/1e6:,.1f}M",    RED),
]
ws1.row_dimensions[3].height = 55
ws1.row_dimensions[4].height = 40
for i, (label, val, colour) in enumerate(kpis, 1):
    col = get_column_letter(i)
    ws1.column_dimensions[col].width = 17
    c = ws1.cell(row=3, column=i, value=label)
    c.font = Font(bold=True, color=WHITE, size=9)
    c.fill = PatternFill('solid', start_color=colour)
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c = ws1.cell(row=4, column=i, value=val)
    c.font = Font(bold=True, size=13, color=colour)
    c.alignment = Alignment(horizontal='center', vertical='center')

for key, anchor in [('price_dist','A6'),('by_loc','G6'),('monthly','A28'),
                     ('type_pie','G28'),('status','A50'),('avg_type','G50')]:
    if key in charts:
        img = XLImage(charts[key])
        img.width = 480; img.height = 240
        ws1.add_image(img, anchor)

# ── Sheet 2: Raw Data ──
ws2 = wb.create_sheet("Raw Data")
write_df(ws2, df.drop(columns=['Price Segment'], errors='ignore'), header_bg=BLUE)

# ── Sheet 3: Filters ──
ws3 = wb.create_sheet("Filters")
ws3['A1'] = "FILTER SUMMARY"
ws3['A1'].font = Font(bold=True, size=13, color=BLUE)
ws3.merge_cells('A1:D1')
for ci, h in enumerate(['Filter Name','Criteria','Record Count','% of Total'], 1):
    c = ws3.cell(row=2, column=ci, value=h)
    c.font = Font(bold=True, color=WHITE)
    c.fill = PatternFill('solid', start_color=TEAL)
    c.alignment = Alignment(horizontal='center')

criteria_map = {
    'Lagos':'Location == Lagos','Abuja':'Location == Abuja',
    'Port Harcourt':'Location == Port Harcourt','Kano':'Location == Kano',
    'Ibadan':'Location == Ibadan','Duplex':'Property Type == Duplex',
    'Flat':'Property Type == Flat','Mansion':'Property Type == Mansion',
    'Bungalow':'Property Type == Bungalow','Studio Apartment':'Property Type == Studio Apartment',
    'Affordable <10M':'Price < NGN 10M','Mid 10M-50M':'NGN 10M <= Price < NGN 50M',
    'Upper 50M-200M':'NGN 50M <= Price < NGN 200M','Luxury >200M':'Price >= NGN 200M',
    'Available':'Status == Available','Sold':'Status == Sold',
    'Rented':'Status == Rented','Under Offer':'Status == Under Offer',
    '1 Bed':'Bedrooms == 1','3+ Bedrooms':'Bedrooms >= 3','5+ Bedrooms':'Bedrooms >= 5',
    '2022 Listings':'Year == 2022','2023 Listings':'Year == 2023','2024 Listings':'Year == 2024',
    'Q1 Listings':'Quarter == 1 (Jan-Mar)','Q2 Listings':'Quarter == 2 (Apr-Jun)',
    'Q3 Listings':'Quarter == 3 (Jul-Sep)','Q4 Listings':'Quarter == 4 (Oct-Dec)',
    'Weekend Listings':'Listed on Saturday or Sunday','Weekday Listings':'Listed Monday to Friday',
    'Has Parking':'Parking Spaces >= 1','No Parking':'Parking Spaces == 0',
    '2+ Parking':'Parking Spaces >= 2','New Build <5yrs':'Property Age < 5 years',
    'Old Build >20yrs':'Property Age > 20 years',
    'Lagos Premium':'Lagos + 3+ Beds + Available + Price <= NGN 100M',
    'Abuja Luxury':'Abuja + Price >= NGN 100M','Large Mansions':'Mansion + Size >= 300sqm',
}
for ri, (name, crit) in enumerate(criteria_map.items(), 3):
    count = len(filters.get(name, pd.DataFrame()))
    pct = f"{count/len(df)*100:.1f}%"
    for ci, val in enumerate([name, crit, count, pct], 1):
        c = ws3.cell(row=ri, column=ci, value=val)
        c.alignment = Alignment(horizontal='center' if ci != 2 else 'left')
        if ri % 2 == 0:
            c.fill = PatternFill('solid', start_color=GREY)
for col, w in zip('ABCD', [24, 40, 16, 14]):
    ws3.column_dimensions[col].width = w

# ── Sheet 4: Analysis ──
ws4 = wb.create_sheet("Analysis")
section_title(ws4, 1,  1, "1. Price Statistics by Location",      BLUE)
write_df(ws4, avg_loc,       start_row=2,  header_bg=BLUE)
section_title(ws4, 14, 1, "2. Price Statistics by Property Type", TEAL)
write_df(ws4, avg_type,      start_row=15, header_bg=TEAL)
section_title(ws4, 26, 1, "3. Listings per Year",                 ORANGE)
write_df(ws4, listings_year, start_row=27, header_bg=ORANGE)
section_title(ws4, 34, 1, "4. Status Breakdown",                  GREEN)
write_df(ws4, status_df,     start_row=35, header_bg=GREEN)
section_title(ws4, 42, 1, "5. Price Segment Breakdown",           PURPLE)
write_df(ws4, segment_df,    start_row=43, header_bg=PURPLE)
section_title(ws4, 51, 1, "6. Price per sqm by Location",         RED)
write_df(ws4, price_sqm,     start_row=52, header_bg=RED)
section_title(ws4, 63, 1, "7. Agent Performance",                 BLUE)
write_df(ws4, agent_df,      start_row=64, header_bg=BLUE)
section_title(ws4, 78, 1, "8. Top 10 Most Expensive Properties",  RED)
write_df(ws4, top10,         start_row=79, header_bg=RED)

# ── Sheet 5: Pivot Tables ──
ws5 = wb.create_sheet("Pivot Tables")
row = 1
def write_pivot(ws, df_p, title, start_row, bg=BLUE):
    section_title(ws, start_row, 1, title, bg)
    write_df(ws, df_p, start_row=start_row+1, header_bg=bg)
    return start_row + len(df_p) + 5
row = write_pivot(ws5, pivot1, "PIVOT 1 - Listing Count: Location x Property Type",   row, BLUE)
row = write_pivot(ws5, pivot2, "PIVOT 2 - Avg Price (NGN): Location x Property Type", row, TEAL)
row = write_pivot(ws5, pivot3, "PIVOT 3 - Listing Count: Year x Quarter",              row, ORANGE)
row = write_pivot(ws5, pivot4, "PIVOT 4 - Avg Price (NGN): Year x Status",             row, GREEN)
row = write_pivot(ws5, pivot5, "PIVOT 5 - Listing Count: Location x Bedrooms",         row, PURPLE)
row = write_pivot(ws5, pivot6, "PIVOT 6 - Listing Count: Property Type x Status",      row, RED)
row = write_pivot(ws5, pivot7, "PIVOT 7 - Avg Price/sqm (NGN): Location x Type",       row, BLUE)

# ── Sheet 6: Date Analysis ──
date_cols = ['Property ID','Listing Date','Day','Day of Week','Week','Month','Month Name',
             'Quarter','Quarter Label','Year','Month Year','Days on Market','Property Age']
ws6 = wb.create_sheet("Date Analysis")
write_df(ws6, df[date_cols], start_row=1, header_bg=PURPLE)

# ── Sheet 7: Charts ──
ws7 = wb.create_sheet("Charts")
ws7.sheet_view.showGridLines = False
ws7['A1'] = "ALL CHARTS - MICOK-HOMES"
ws7['A1'].font = Font(bold=True, size=14, color=BLUE)
positions = ['A2','J2','A28','J28','A54','J54','A80','J80',
             'A106','J106','A132','J132','A158','J158']
for key, pos in zip(charts.keys(), positions):
    img = XLImage(charts[key])
    img.width = 480; img.height = 250
    ws7.add_image(img, pos)

# ── Sheet 8: Top 10 ──
ws8 = wb.create_sheet("Top 10 Expensive")
write_df(ws8, top10, header_bg=RED)

# ── SAVE ──
wb.save(OUT_XLSX)
print(f"\n Excel file saved: {OUT_XLSX}")
print("   Open it in Excel or upload to Google Sheets to view your dashboard!")

##  Done!

Your Excel file `Micok-Homes_Analysis.xlsx` has been saved in this folder.

**It contains 8 sheets:**
| Sheet | Contents |
|-------|----------|
|  Dashboard | KPI cards + 6 embedded charts |
|  Raw Data | Full cleaned dataset |
|  Filters | All 38 filter summaries |
|  Analysis | Price stats, agent performance, top 10 |
|  Pivot Tables | 7 pivot tables |
|  Date Analysis | Date features for every listing |
|  Charts | All 14 charts on one sheet |
|  Top 10 Expensive | Top 10 most expensive properties |

*To re-run with updated data: replace `Micok-Homes_Dataset.xlsx` and run all cells again (Cell menu → Run All)*
